# features

> Continuous measurements for learned scoring

In [1]:
#| default_exp features

In [2]:
#| hide
from nbdev.showdoc import *

The rules keep only what crosses a bar, and a learned scorer wants the bar removed. This module measures prose as continuous numbers: every rule's findings as a rate, the sentence and paragraph distributions the thresholds truncate, the parts-of-speech profile, specifics density, referential load, and burstiness. Three of these measures failed earlier as threshold rules, and the para and score notebooks record why. As features they need no threshold, because a learned combiner such as a random forest weighs them against labeled examples. The score stays rule-based and explainable. The feature vector exists for calibration experiments, and anything the experiments validate can graduate into a rule.

In [3]:
#| export
import statistics
from collections import Counter
from fastcore.utils import *
from slopometer.core import *
from slopometer.segment import *
from slopometer.lexicon import *
from slopometer.syntax import *
from slopometer.para import *
from slopometer.score import run_rules

In [4]:
from fastcore.test import *
from nbdev.config import get_config

## Burstiness

Sentence-length variation has a direct measure from the temporal-patterns literature: the [Goh-Barabási burstiness parameter](https://arxiv.org/abs/physics/0610233), B = (σ − μ)/(σ + μ) over the lengths. Uniform text sits near −1, random variation near 0, heavy mixing above. Low burstiness is the classic signature of machine text, and it is also correct reference prose, which is why this is a feature and not a rule: the register decides its polarity, and the calibration passage is the most uniform document we measure. `flattest_window` localizes the number for reports, returning the run of sentences where lengths vary least, which paired with a specifics count separates dense contract prose from chopped emptiness.

In [5]:
#| export
def burstiness(xs):
    "Goh-Barabasi B over `xs`: -1 uniform, 0 random, positive bursty"
    if len(xs) < 2: return 0.0
    m, s = statistics.mean(xs), statistics.stdev(xs)
    return 0.0 if m + s == 0 else round((s - m) / (s + m), 3)

def flattest_window(
    xs, # Sentence lengths, in document order
    k=5, # Window size in sentences
):
    "`(start, spread)` of the length-`k` run with the lowest variation, for localizing monotone stretches"
    if len(xs) <= k: return 0, round(statistics.stdev(xs), 2) if len(xs) > 1 else 0.0
    spans = [(i, round(statistics.stdev(xs[i:i+k]), 2)) for i in range(len(xs) - k + 1)]
    return min(spans, key=lambda t: t[1])

In [6]:
uniform = [10, 10, 11, 10, 10, 11, 10]
mixed = [3, 24, 8, 31, 5, 19, 12]
test_eq(burstiness(uniform) < -0.7, True)
test_eq(burstiness(mixed) > burstiness(uniform), True)
flattest_window([20, 21, 20, 3, 30, 7, 28, 4], k=3), round(burstiness(mixed), 2)

((0, 0.58), -0.17)

## The feature vector

`features` reduces a document to one flat dict. The first group restates the rules as rates: each rule's weighted findings per 100 words, plus their total, which equals the report's density. The rest measure what no threshold sees. Distribution features carry the sentence and paragraph shapes. The POS profile carries adverb and adjective load, noun-to-verb balance, and copula share. Specifics density counts numerals, code spans, and proper nouns per 100 words, which is tell 26's "depth is specifics" made continuous, and the theory2 detector the rules lack. Referential load returns from the score notebook's prototype: pronoun-subject share and cold-definite share, failed rules and welcome features. Connective rates close the set. Names stay short because they become dataframe columns.

In [7]:
#| export
_conn = dict(add={'also', 'furthermore', 'moreover', 'additionally'}, causal={'because', 'therefore', 'thus', 'hence'},
    advers={'but', 'however', 'although', 'though', 'yet'})

def features(txt):
    "Continuous measurements of markdown `txt`, one flat dict"
    blocks = segment(txt)
    docs = parse_blocks(blocks)
    prose = [d for d in docs if d is not None]
    sents = [s for d in prose for s in d.sents]
    toks = [t for d in prose for t in d]
    nw = max(sum(1 for t in toks if t.is_alpha), 1)
    res = dict(words=nw, sents=len(sents), headings=n_headings(blocks))
    agg = Counter()
    for f in run_rules(txt): agg[f.rule] += f.weight
    res |= {f'r_{k}': round(100*v/nw, 2) for k, v in agg.items()}
    res['density'] = round(100*sum(agg.values())/nw, 1)
    slens = [sum(1 for t in s if not t.is_punct) for s in sents]
    if len(slens) > 1:
        res |= dict(slen_mean=round(statistics.mean(slens), 1), slen_max=max(slens), slen_burst=burstiness(slens))
        starts = [next((t.lemma_.lower() for t in s if t.is_alpha), '') for s in sents]
        res['start_div'] = round(len(set(starts))/len(starts), 2)
    plens = [sum(1 for _ in d.sents) for d in prose]
    if plens: res |= dict(psents_mean=round(statistics.mean(plens), 1), psents_max=max(plens))
    pos = Counter(t.pos_ for t in toks)
    res |= dict(adv=round(100*pos['ADV']/nw, 1), adj=round(100*pos['ADJ']/nw, 1),
        noun_verb=round(pos['NOUN']/max(pos['VERB'], 1), 2),
        cop=round(sum(1 for s in sents if s.root.lemma_ == 'be')/max(len(sents), 1), 2))
    ncode = sum(len(re.findall(r'X{2,}', scrub(b.txt))) for b in blocks)
    res['specifics'] = round(100*(pos['NUM'] + pos['PROPN'] + ncode)/nw, 1)
    res['pron_subj'] = round(sum(1 for s in sents if any(t.dep_ in ('nsubj', 'nsubjpass') and t.pos_ == 'PRON'
        and t.head.dep_ == 'ROOT' for t in s))/max(len(sents), 1), 2)
    seen, ndef, ncold = set(), 0, 0
    for d in prose:
        for t in d:
            if t.pos_ not in ('NOUN', 'PROPN') or len(set(t.lower_)) == 1: continue
            lem = t.lemma_.lower()
            kids = [c.lower_ for c in t.children]
            if 'the' in kids and 'same' not in kids:
                ndef += 1
                if lem not in seen: ncold += 1
            seen.add(lem)
    res |= dict(defs=ndef, cold=round(ncold/max(ndef, 1), 2))
    res |= {f'conn_{k}': round(100*sum(1 for t in toks if t.lemma_.lower() in v)/nw, 2) for k, v in _conn.items()}
    return res

In [8]:
sd = get_config().config_path/'samples'
texts = {f't{i}': (sd/f'theory{i}.md').read_text() for i in (1, 2, 3, 4)}
fx = {k: features(t) for k, t in texts.items()}
cols = ['density', 'slen_mean', 'slen_burst', 'start_div', 'psents_max', 'specifics', 'pron_subj', 'cold', 'cop']
test_eq(fx['t2']['specifics'] < fx['t4']['specifics'], True)
{k: {c: v[c] for c in cols if c in v} for k, v in fx.items()}

{'t1': {'density': 59.6,
  'slen_mean': 21.9,
  'slen_burst': -0.047,
  'start_div': 0.7,
  'psents_max': 5,
  'specifics': 3.4,
  'pron_subj': 0.09,
  'cold': 0.56,
  'cop': 0.35},
 't2': {'density': 5.4,
  'slen_mean': 10.9,
  'slen_burst': -0.336,
  'start_div': 0.48,
  'psents_max': 6,
  'specifics': 2.8,
  'pron_subj': 0.06,
  'cold': 0.46,
  'cop': 0.21},
 't3': {'density': 14.0,
  'slen_mean': 14.8,
  'slen_burst': -0.23,
  'start_div': 0.54,
  'psents_max': 7,
  'specifics': 2.5,
  'pron_subj': 0.03,
  'cold': 0.28,
  'cop': 0.09},
 't4': {'density': 1.5,
  'slen_mean': 12.4,
  'slen_burst': -0.26,
  'start_div': 0.61,
  'psents_max': 8,
  'specifics': 9.3,
  'pron_subj': 0.03,
  'cold': 0.3,
  'cop': 0.03}}

The columns tell four stories. Specifics density separates the pair nothing else could: theory2 and theory4 both score low on rules, and theory4 carries more than triple the specifics (9.3 against 2.8), which is the concrete-versus-empty distinction measured at last. Burstiness confirms its prediction: theory2 is the most uniform draft at −0.34, with the lowest opener diversity. The copula rate is the surprise: it falls monotonically through the whole ladder, 0.35 to 0.21 to 0.09 to 0.03, making "is"-heavy definitional prose the cleanest single tracker of the slop-to-reference progression we have measured. And pronoun-subject share halves from slop to reference. None of these carries a weight yet. They wait for labels, and the forest experiment decides which of them earn one.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()